# Test the jar flows (6 cases)

Six cells in a row, each one a real call with the real model and the live jars of `CIF_0001` from `pfm` (needs network, `SERVER_API_KEY` and
`AI_PLATFORM_API_KEY` in `../.env`). Run the setup cell once, then any case cell on its own (each takes ~10-40 s). **Restart the kernel first** if
you ran an older notebook in it. Contract: `docs/jars/`.

| # | Case | Expect |
|---|---|---|
| 1 | `POST /jar-distribute` 500.000đ | `distribute_amount`, `check_distribute` has no hard error |
| 2 | `POST /jar-rebalance` 500.000đ from the jar holding the most | `rebalance_jars`, `check_rebalance` has no hard error |
| 3 | chat: "Chuyển 300 nghìn vào hũ Tiết kiệm" | `distribute_amount` (a jar with no balance yet can receive) |
| 4 | chat: "Chia 1 triệu vào A 600k và B 400k" | ONE `distribute_amount` with the 2 jars |
| 5 | chat: "Chuyển 300 nghìn từ hũ A sang hũ B và 200 nghìn sang hũ C" | `rebalance_jars` (`from_jar_id` + 2 `allocations`) |
| 6 | chat: more than the unallocated money | **no `ui`**, the answer says how much is left |

The model is non-deterministic: a hard error from the checks fails the cell, anything softer is printed.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")

assert os.environ.get("SERVER_API_KEY"), f"Set SERVER_API_KEY in {REPO_ROOT / '.env'} first"

from fastapi.testclient import TestClient
from langgraph.checkpoint.memory import InMemorySaver

import server
from jar_checks import check_distribute, check_rebalance, pool_available, source_available
from tools.customer_data_tools import fetch_jar_summary

# In-memory agent: every run starts a fresh thread. TestClient goes through the real routes, so the validators and ui_guard run.
server.agent = server.build_agent(checkpointer=InMemorySaver())
client = TestClient(server.app)
HEADERS = {"X-API-Key": os.environ["SERVER_API_KEY"]}
CUSTOMER = "CIF_0001"


def vnd(amount: int) -> str:
    return f"{amount:,}".replace(",", ".") + "đ"


def chat(message: str, thread: str) -> dict:
    response = client.post("/chat", headers=HEADERS, json={"message": message, "user_id": CUSTOMER, "thread_id": thread})
    assert response.status_code == 200, response.text
    body = response.json()
    print(f"Q: {message}\nANSWER:\n{body['answer']}\n\nUI:\n{body['ui']}\n" + "-" * 60)
    return body


# The live jars decide the questions, so the cells keep working when the pfm data changes.
summary = fetch_jar_summary(CUSTOMER)
jars = summary["jars"]
funded = [j for j in jars if j.get("balance") is not None]
giver = max(funded, key=lambda j: j["spendable"] or 0)  # the jar that holds the most
others = [j for j in jars if j["id"] != giver["id"]]
print("unallocated:", vnd(pool_available(summary)))
for j in jars:
    print(f"  {j['label']:<12} id={j['id']:<11} limit={j['limit']} balance={j['balance']} spendable={j['spendable']}")

/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


unallocated: 13.244.000đ
  Thiết yếu    id=essentials  limit=9600000 balance=1651000 spendable=1651000
  Ăn uống      id=food        limit=4800000 balance=2613000 spendable=2613000
  Di chuyển    id=transport   limit=1800000 balance=1096000 spendable=1096000
  Hưởng thụ    id=lifestyle   limit=3000000 balance=1796000 spendable=1796000
  Sức khỏe     id=health      limit=1200000 balance=1200000 spendable=1200000
  Tiết kiệm    id=savings     limit=None balance=None spendable=None


In [2]:
# 1. POST /jar-distribute -- share 500.000đ of the unallocated money; the agent picks the jars
AMOUNT = 500_000
response = client.post("/jar-distribute", headers=HEADERS, json={"user_id": CUSTOMER, "amount": AMOUNT})
assert response.status_code == 200, response.text
body = response.json()
print("ANSWER:\n", body["answer"], "\n\nUI:\n", body["ui"])
if body["ui"] is None:
    print("\n(no ui: valid when the unallocated money is below the amount -- the answer says so)")
else:
    assert body["ui"]["type"] == "distribute_amount", body["ui"]["type"]
    verdict = check_distribute(summary, AMOUNT, body["ui"])
    assert verdict.hard == [], verdict.hard
    print("soft notes:", verdict.soft)

ANSWER:
 Đề xuất chia 500.000đ từ tiền chưa phân bổ vào hai hũ chi nhiều nhất.

Bù 350.000đ cho hũ Thiết yếu và 150.000đ cho hũ Ăn uống. 

UI:
 {'type': 'distribute_amount', 'allocations': [{'to_jar_id': 'essentials', 'amount': 350000}, {'to_jar_id': 'food', 'amount': 150000}], 'reason': 'Thiết yếu chi trung bình ~8 triệu/tháng, số dư chỉ 1,65 triệu nên cần bù nhiều nhất. Ăn uống chi trung bình ~3,2 triệu/tháng, số dư 2,6 triệu cũng khá thấp nên nhận phần còn lại.'}
soft notes: []


In [ ]:
# 2. POST /jar-rebalance -- move 500.000đ out of the jar that holds the most; the agent picks the receivers
amount = min(500_000, source_available(summary, giver["id"]))
response = client.post("/jar-rebalance", headers=HEADERS, json={"user_id": CUSTOMER, "from_jar_id": giver["id"], "amount": amount})
assert response.status_code == 200, response.text
body = response.json()
print(f"from {giver['label']} ({vnd(giver['spendable'])}), amount {vnd(amount)}\nANSWER:\n", body["answer"], "\n\nUI:\n", body["ui"])
if body["ui"] is not None:
    assert body["ui"]["type"] == "rebalance_jars", body["ui"]["type"]
    assert body["ui"]["from_jar_id"] == giver["id"], "the giver comes from the request, never from the model"
    verdict = check_rebalance(summary, giver["id"], amount, body["ui"])
    assert verdict.hard == [], verdict.hard
    print("soft notes:", verdict.soft)

In [ ]:
# 3. chat: "Chuyển 300 nghìn vào hũ Tiết kiệm" -- skill distribute-amount; the jar has no balance yet and may still receive
ui = chat("Chuyển 300 nghìn vào hũ Tiết kiệm giúp tôi.", "jar-3")["ui"]
if ui is None:
    print("(no ui: valid only when the unallocated money is below 300.000đ)")
else:
    assert ui["type"] == "distribute_amount", ui["type"]
    verdict = check_distribute(summary, 300_000, ui)
    assert verdict.hard == [], verdict.hard
    print("allocations:", ui["allocations"], "| soft notes:", verdict.soft)

In [ ]:
# 4. chat: two jars in one sentence -> ONE distribute_amount with both jars
a, b = jars[1], jars[2]
ui = chat(f"Chia 1 triệu vào hũ {a['label']} 600 nghìn và hũ {b['label']} 400 nghìn.", "jar-4")["ui"]
if ui is None:
    print("(no ui: valid only when the unallocated money is below 1.000.000đ)")
else:
    assert ui["type"] == "distribute_amount", ui["type"]
    verdict = check_distribute(summary, 1_000_000, ui)
    assert verdict.hard == [], verdict.hard
    print("allocations:", ui["allocations"], "(expected", a["id"], "600000 and", b["id"], "400000 )")

In [ ]:
# 5. chat: one giving jar, two receivers -- skill rebalance-jars
r1, r2 = others[0], others[1]
ui = chat(f"Chuyển 300 nghìn từ hũ {giver['label']} sang hũ {r1['label']} và 200 nghìn sang hũ {r2['label']}.", "jar-5")["ui"]
assert ui is not None and ui["type"] == "rebalance_jars", "expected a rebalance_jars form"
assert ui["from_jar_id"] == giver["id"] and len(ui["allocations"]) == 2, ui
verdict = check_rebalance(summary, giver["id"], 500_000, ui)
assert verdict.hard == [], verdict.hard
print("allocations:", ui["allocations"], "| soft notes:", verdict.soft)

In [ ]:
# 6. chat: more than the unallocated money -> no form, the answer says how much is left
too_much = pool_available(summary) + 1_000_000
ui = chat(f"Chuyển {vnd(too_much)} vào hũ {jars[1]['label']}.", "jar-6")["ui"]
assert ui is None, "money that is not there must never come back as a distribute_amount form"
print("OK: no ui. Unallocated money is", vnd(pool_available(summary)))